In [1]:
from ollama import chat
from ollama import ChatResponse
prompt = "What is the capital of France?"

response: ChatResponse = chat(model='llama3.2', messages = [
    {
        'role': 'user',
        'content': prompt
    },
])

print(response.message.content)

The capital of France is Paris.


## roles
--> User (the one interacting with the assistant (the model))
--> assistant (the model)
--> system (tells the assistant (the model) on how to work and what to work like commanding it to do something)

In [ ]:
from ollama import chat

response = chat(
  model='llama3.2',
  messages=[{'role': 'user', 'content': 'How many letter r are in strawberry?'}],
  stream=False
)

print('Answer:\n', response.message.content)

Answer:
 There is one letter "R" in the word "strawberry".


In [ ]:
from ollama import chat
from ollama import ChatResponse
prompt = "What is (28*28)+76?"

response: ChatResponse = chat(model='llama3.2', messages = [
    {
        'role': 'user',
        'content': prompt,
    },
])

print(response.message.content)


To calculate this, I'll follow the order of operations:

1. Multiply 28 and 28:
28 * 28 = 784
2. Add 76 to the result:
784 + 76 = 860

The final answer is: 860


## Project - 1 (Smart CLI chatbot that does math, fetch latest weather, search wikipedia and look for movies with specific genre)

 1. Build the tools that the agent will call

--> Weather api calling

In [ ]:
from geopy.geocoders import Nominatim
import openmeteo_requests

openmeteo = openmeteo_requests.Client()

def get_weather(city: str) -> str:
    geolocator = Nominatim(user_agent="weather_app")
    location = geolocator.geocode(city)

    if not location:
        return f"Could not find the location: {city}"
    
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location.latitude,
        "longitude": location.longitude,
        "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
	    "current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature"],
        "timezone": "auto"
    }


    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]

        current = response.Current()
        current_temp = current.Variables(0).Value()
        current_humidity = current.Variables(1).Value()
        apparent_temp = current.Variables(2).Value()

        return (
            f"Weather for {location.address}:\n"
            f"- Temperature: {current_temp:.1f}°C\n"
            f"- Feels Like: {apparent_temp:.1f}°C\n"
            f"- Humidity: {current_humidity:.0f}%"
        )
    

    except Exception as e:
        return f"Error fetching weather data: {e}"

In [ ]:
print(get_weather("Stockholm"))

Weather for Stockholm, Stockholms kommun, Stockholms län, 111 29, Sverige:
- Temperature: 19.1°C
- Feels Like: 16.8°C
- Humidity: 48%


--> wikipedia api

In [ ]:
import wikipediaapi
def search_wikipedia(query: str) -> str:
    wiki = wikipediaapi.Wikipedia(user_agent = "wiki_search_app", language = 'en')
    page = wiki.page(query)
    if page.exists():
        return f"Title: {page.title}\n Summary: {page.summary[:500]}..."

In [ ]:
print(search_wikipedia("Rodents"))

Title: Rodent
 Summary: Rodents (from Latin rodens, 'gnawing') are a group of mammals belonging to the order Rodentia ( roh-DEN-shə or roh-DEN-chə) characterized by a single pair of continuously growing incisors in each of the upper and lower jaws. Rodents make up about 40% of all mammal species. They are native to all major landmasses except Antarctica and several oceanic islands, though they have subsequently been introduced to most of these landmasses by human activity. Most rodents are small animals with robust bod...


--> movie api calling based on genre

In [ ]:
import dotenv
import requests
dotenv.load_dotenv()
api_key = dotenv.get_key(".env", "TMDB_API")

def get_movie_recommendations(genre: str):
    url = "https://api.themoviedb.org/3"
    genre_url = f"{url}/genre/movie/list"
    params = {"api_key": api_key, "language": "en-US"}

    genre_response = requests.get(genre_url, params = params)
    if genre_response.status_code != 200:
        return f"Error fetching genres: {genre_response.status_code}"

    genres = genre_response.json().get("genres", [])
    genre_id = next((g["id"] for g in genres if g["name"].lower() == genre.lower()), None)
    if not genre_id:
        available_genres = ", ".join([g["name"] for g in genres])
        return f"Genre '{genre}' not found."
    
    discover_url = f"{url}/discover/movie"
    discover_params = {
        "api_key": api_key,
        "language": "en-US",
        "sort_by" : "popularity.desc",
        "with_genres": genre_id,
        "page": 1
    }

    movie_response = requests.get(discover_url, params = discover_params)
    if movie_response.status_code != 200:
        return f"Error fetching movies: {movie_response.status_code}"
    
    recommendations = []

    movie_data = movie_response.json().get("results", [])

    for movie in movie_data[:5]:
        recommendations.append({
            "title": movie.get("title"),
            "rating": movie.get("vote_average"),
            "release_date": movie.get("release_date"),
            "overview": movie.get("overview")
        })
    
    return recommendations


GENRE = "Horror"
results = get_movie_recommendations(GENRE)
if isinstance(results, list):
    print(f"--- Top 5 Recommendations for {GENRE} ---")
    for idx, movie in enumerate(results, 1):
        print(f"{idx}. {movie['title']} ({movie['release_date'][:4]})")
        print(f"   Rating: {movie['rating']}/10")
        print(f"   Overview: {movie['overview']}\n")
else:
    print(results)

--- Top 5 Recommendations for Horror ---
1. Obsession (2026)
   Rating: 7.9/10
   Overview: After breaking the mysterious "One Wish Willow" to win his crush's heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.

2. Lee Cronin's The Mummy (2026)
   Rating: 8.078/10
   Overview: The young daughter of a journalist disappears into the desert without a trace—eight years later, the broken family is shocked when she is returned to them, as what should be a joyful reunion turns into a living nightmare.

3. Backrooms (2026)
   Rating: 6.8/10
   Overview: A strange doorway appears in the basement of a furniture showroom.

4. Hokum (2026)
   Rating: 6.7/10
   Overview: When novelist Ohm Bauman retreats to a remote inn to scatter his parents' ashes, he is consumed by tales of a witch haunting the honeymoon suite. Disturbing visions and a shocking disappearance forces him to confront dark corners of his past

In [ ]:
def get_similar_and_recommend_movies(movie):
    url = " https://api.themoviedb.org/3"
    params = {"api_key": api_key, "language": "en-US"}

    search_url = f"{url}/search/movie"
    search_params = {**params, "query": movie, "page": 1}

    search_response = requests.get(search_url, params = search_params)
    if search_response.status_code != 200:
        return f"Error searching for movie: {search_response.status_code}"
    
    results = search_response.json().get("results", [])
    if not results:
        return f"No results found for '{movie}'"
    
    matched_movie = results[0]
    movie_id = matched_movie["id"]
    actual_title = matched_movie["title"]
    release_year = matched_movie.get("release_date", "----")[:4]

    print(f"Found Match: '{actual_title}' ({release_year}) -> ID: {movie_id}\n")

    similar_url = f"{url}/movie/{movie_id}/similar"
    similar_response = requests.get(similar_url, params = params)
    if similar_response.status_code != 200:
        return f"Error fetching similar movies: {similar_response.status_code}"
    similar_res = similar_response.json().get("results", [])

    reco_url = f"{url}/movie/{movie_id}/recommendations"
    reco_res = requests.get(reco_url, params=params).json().get("results", [])

    similar_list = [f"{m['title']} ({m.get('release_date', '----')[:4]})" for m in similar_res[:5]]
    
    reco_list = [f"{m['title']} ({m.get('release_date', '----')[:4]})" for m in reco_res[:5]]
    
    return {
        "source_movie": f"{actual_title} ({release_year})",
        "similar_by_genre": similar_list,
        "recommended_by_users": reco_list
    }


output = get_similar_and_recommend_movies("Backrooms")
if isinstance(output, dict):
    print(f"=== Structural Matches (Similar Genres/Keywords) ===")
    for idx, movie in enumerate(output["similar_by_genre"], 1):
        print(f" {idx}. {movie}")
        
    print(f"\n=== Algorithmic Matches (Users Who Liked '{output['source_movie']}' Also Watched) ===")
    for idx, movie in enumerate(output["recommended_by_users"], 1):
        print(f" {idx}. {movie}")
else:
    print(output)



Found Match: 'Backrooms' (2026) -> ID: 1083381

=== Structural Matches (Similar Genres/Keywords) ===
 1. The Recall (2017)
 2. Troll 2 (1990)
 3. Haunted Hospital: Heilstatten (2018)
 4. Gamera vs. Guiron (1969)
 5. The Last Voyage of the Demeter (2023)

=== Algorithmic Matches (Users Who Liked 'Backrooms (2026)' Also Watched) ===
 1. Phoenix Forgotten (2017)
 2. undertone (2026)
 3. The Entity (1982)
 4. Event Horizon (1997)
 5. The Conjuring: Last Rites (2025)


2. Define a schema (in json) that will define these tool to be called

In [ ]:
import tools

AVAILABLE_TOOLS = {
    'get_weather': get_weather,
    'search_wikipedia': search_wikipedia,
    'get_movie_recommendations': get_movie_recommendations,
    'get_similar_and_recommend_movies': get_similar_and_recommend_movies
}


TOOL_DEFINITIONS = [
    {
        'type': 'function',
        'function' : {
            'name': 'get_weather',
            'description' : 'Get current weather information for a specified city.',
            'parameters' : {
                'type': 'object',
                'properties': {'city': {'type': 'string', 'description': 'The city name, e.g., Paris, Stockholm'}},
                'required': ['city'],
            },
        },
    },

    {
        'type': 'function',
        'function': {
            'name': 'search_wikipedia',
            'description': 'Search Wikipedia to get factual descriptions about people, historical events, places, or concepts.',
            'parameters': {
                'type': 'object',
                'properties': {'query': {'type': 'string', 'description': 'The search term or topic'}},
                'required': ['query'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_movie_recommendations',
            'description': 'Get top movie recommendations based on a specified genre.',
            'parameters': {
                'type': 'object',
                'properties': {'genre': {'type': 'string', 'description': 'The movie genre, e.g., Action, Comedy, Horror'}},
                'required': ['genre'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_similar_and_recommend_movies',
            'description': "Get movies similar to a specified movie based on genres/keywords and user behavior.",
            'parameters': {
                'type': 'object',
                'properties': {'movie': {'type': 'string', 'description': "The movie title to find similarities and recommendations for"}},
                'required': ['movie'],
            },
        },
    }
]

3. Call the main chatbot loop

In [ ]:
from ollama import chat

def run_chatbot():
    print("==================================================")
    print("🤖 Smart CLI Chatbot Initialized (Llama 3.2 Agent)")
    print("Tools loaded: Math, Weather, Wikipedia, Movies")
    print("Type 'exit' or 'quit' to end the conversation.")
    print("==================================================\n")

    conversational_history = [
        {"role": "system", "content" : "You are a helpful CLI assistant. You have access to tools for weather, wikipedia searches, and movie recommendations. Use them whenever necessary to provide accurate information."}
    ]

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day!")
            break

        if not user_input.strip():
            continue
            
        conversational_history.append({"role": "user", "content": user_input})

        response = chat(
            model = "llama3.2",
            messages = conversational_history,
            tools = TOOL_DEFINITIONS
        )

        if response.message.tool_calls:
            conversational_history.append(response.message)

            for tool in response.message.tool_calls:
                name = tool.function.name
                args = tool.function.arguments

                print(f"\n⚙️  [Agent Logic] Invoking tool '{name}' with arguments: {args}")

                if name in AVAILABLE_TOOLS:
                    tool_result = AVAILABLE_TOOLS[name](**args)
                    print(f"🔌 [Tool Output] Derived data: {tool_result}")

                    conversational_history.append({
                        "role": "tool",
                        "content" : str(tool_result),
                        "name": name
                    })
            final_response = chat(model = "llama3.2", messages = conversational_history)
            print(f"\n🤖 Chatbot: {final_response.message.content}")
            conversational_history.append(final_response.message)
        else:
            print(f"\n🤖 Chatbot: {response.message.content}")
            conversational_history.append(response.message)
    
if __name__ == "__main__":
    run_chatbot()


🤖 Smart CLI Chatbot Initialized (Llama 3.2 Agent)
Tools loaded: Math, Weather, Wikipedia, Movies
Type 'exit' or 'quit' to end the conversation.

Chatbot: Goodbye! Have a great day!


--> Activate environment:- .\winenv\bin\Activate.ps1

## Project - 2 (Extension) --> Add memory to the project with Langchain

In [34]:
from langchain.tools import tool
from geopy.geocoders import Nominatim
import openmeteo_requests

openmeteo = openmeteo_requests.Client()

@tool
def get_weather(city: str) -> str:
    """Fetch current weather data for a given city name.
    Args:
        city: The name of the city (e.g., 'London', 'Tokyo').
    """
    
    geolocator = Nominatim(user_agent="weather_app")
    location = geolocator.geocode(city)

    if not location:
        return f"Could not find the location: {city}"
    
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location.latitude,
        "longitude": location.longitude,
        "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
	    "current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature"],
        "timezone": "auto"
    }


    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]

        current = response.Current()
        current_temp = current.Variables(0).Value()
        current_humidity = current.Variables(1).Value()
        apparent_temp = current.Variables(2).Value()

        return (
            f"Weather for {location.address}:\n"
            f"- Temperature: {current_temp:.1f}°C\n"
            f"- Feels Like: {apparent_temp:.1f}°C\n"
            f"- Humidity: {current_humidity:.0f}%"
        )
    

    except Exception as e:
        return f"Error fetching weather data: {e}"

In [35]:
import wikipediaapi
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia to get factual descriptions about people, historical events, places, or concepts.
    Args:        query: The search term or topic to look up on Wikipedia.
    """
    wiki = wikipediaapi.Wikipedia(user_agent = "wiki_search_app", language = 'en')
    page = wiki.page(query)
    if page.exists():
        return f"Title: {page.title}\n Summary: {page.summary[:500]}..."

In [37]:
import dotenv
import requests
dotenv.load_dotenv()
api_key = dotenv.get_key(".env", "TMDB_API")

@tool
def get_movie_recommendations(genre: str):
    """Get top movie recommendations based on a specified genre.
    Args:        genre: The movie genre to search for (e.g., Action, Comedy, Horror)."""
    url = "https://api.themoviedb.org/3"
    genre_url = f"{url}/genre/movie/list"
    params = {"api_key": api_key, "language": "en-US"}

    genre_response = requests.get(genre_url, params = params)
    if genre_response.status_code != 200:
        return f"Error fetching genres: {genre_response.status_code}"

    genres = genre_response.json().get("genres", [])
    genre_id = next((g["id"] for g in genres if g["name"].lower() == genre.lower()), None)
    if not genre_id:
        available_genres = ", ".join([g["name"] for g in genres])
        return f"Genre '{genre}' not found."
    
    discover_url = f"{url}/discover/movie"
    discover_params = {
        "api_key": api_key,
        "language": "en-US",
        "sort_by" : "popularity.desc",
        "with_genres": genre_id,
        "page": 1
    }

    movie_response = requests.get(discover_url, params = discover_params)
    if movie_response.status_code != 200:
        return f"Error fetching movies: {movie_response.status_code}"
    
    recommendations = []

    movie_data = movie_response.json().get("results", [])

    for movie in movie_data[:5]:
        recommendations.append({
            "title": movie.get("title"),
            "rating": movie.get("vote_average"),
            "release_date": movie.get("release_date"),
            "overview": movie.get("overview")
        })
    
    return recommendations

In [39]:

@tool
def get_similar_and_recommend_movies(movie):
    """Get movies similar to a specified movie based on genres/keywords and user behavior.
    Args:        movie: The movie title to find similarities and recommendations for."""
    url = " https://api.themoviedb.org/3"
    params = {"api_key": api_key, "language": "en-US"}

    search_url = f"{url}/search/movie"
    search_params = {**params, "query": movie, "page": 1}

    search_response = requests.get(search_url, params = search_params)
    if search_response.status_code != 200:
        return f"Error searching for movie: {search_response.status_code}"
    
    results = search_response.json().get("results", [])
    if not results:
        return f"No results found for '{movie}'"
    
    matched_movie = results[0]
    movie_id = matched_movie["id"]
    actual_title = matched_movie["title"]
    release_year = matched_movie.get("release_date", "----")[:4]

    print(f"Found Match: '{actual_title}' ({release_year}) -> ID: {movie_id}\n")

    similar_url = f"{url}/movie/{movie_id}/similar"
    similar_response = requests.get(similar_url, params = params)
    if similar_response.status_code != 200:
        return f"Error fetching similar movies: {similar_response.status_code}"
    similar_res = similar_response.json().get("results", [])

    reco_url = f"{url}/movie/{movie_id}/recommendations"
    reco_res = requests.get(reco_url, params=params).json().get("results", [])

    similar_list = [f"{m['title']} ({m.get('release_date', '----')[:4]})" for m in similar_res[:5]]
    
    reco_list = [f"{m['title']} ({m.get('release_date', '----')[:4]})" for m in reco_res[:5]]
    
    return {
        "source_movie": f"{actual_title} ({release_year})",
        "similar_by_genre": similar_list,
        "recommended_by_users": reco_list
    }


In [40]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "ollama:llama3.2",
    temperature = 0.5,
)

In [41]:
my_tools = [get_weather, search_wikipedia, get_movie_recommendations, get_similar_and_recommend_movies]

In [55]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent = create_agent(
    model = "ollama:llama3.2",
    tools = my_tools,
    system_prompt = "You are a helpful CLI assistant. You have access to tools for weather, wikipedia searches, and movie recommendations. Use them whenever necessary to provide accurate information.",
    checkpointer = memory
)




For modern agents built in the LangChain ecosystem, using InMemorySaver with LangGraph is the vastly superior and recommended approach.

Here is why:-

--> ConversationBufferMemory is a legacy LangChain abstraction that stores all messages in a simple list. It bloats your token usage over long conversations and doesn't handle interruptions or complex agent loops (like "human-in-the-loop").


--> InMemorySaver (used in langgraph) is a checkpointer used by LangGraph to capture the exact state of your multi-agent workflow at every step. It provides automatic thread continuity, state resumption, and lets you use advanced features like time-travel.

--> RunnableWithMessageHistory:- This is the current, standard way to handle memory in LangChain Expression Language (LCEL). It wraps your agent run chain and dynamically injects history based on a session ID.

However, I don't need anything to store it since, it does it itself

In [61]:
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(configurable={"thread_id": "user_session_1"})


print("--- First Turn ---")
response1 = agent.invoke(
    {"messages": [("user", "What is the weather like in Tokyo?")]},
    config=config
)
print("Bot:", response1["messages"][-1].content)


print("\n--- Second Turn ---")
response2 = agent.invoke(
    {"messages": [("user", "Wow, that's warm! I forgot what city we were talking about")]},
    config=config
)
print("Bot:", response2["messages"][-1].content)

print("---------------Test---------------")
print(response2)

--- First Turn ---
Bot: We're back to discussing Tokyo's weather again! The temperature is still 20.6°C (69.1°F) and feels like 23.4°C (74.1°F), with high humidity at 87%. It's a warm and humid day in Tokyo.

If you're looking for something else, feel free to ask or change the topic. I'm here to help!

--- Second Turn ---
Bot: The answer is still not available from the Wikipedia search. We've been discussing Tokyo's weather for a while now!

To recap, we talked about Tokyo's warm and humid weather earlier. The temperature is 20.6°C (69.1°F) and feels like 23.4°C (74.1°F), with high humidity at 87%. If you want to plan your day or ask another question, feel free to ask!
---------------Test---------------
{'messages': [HumanMessage(content='What is the weather like in Tokyo?', additional_kwargs={}, response_metadata={}, id='0049e7b2-5e07-4995-94aa-91828996d2a8'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-06-11T15:03:29.5525217

In [ ]:
from langchain_core.runnables import RunnableConfig

def run_chatbot():
    print("==================================================")
    print("🤖 Smart CLI Chatbot Initialized (LangChain + Llama 3.2)")
    print("Tools managed automatically: Math, Weather, Wikipedia, Movies")
    print("Type 'exit' or 'quit' to end the conversation.")
    print("==================================================\n")

    config = RunnableConfig(configurable={"thread_id": "cli_session_1"})

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day!")
            break

        if not user_input.strip():
            continue
            
        response = agent.invoke(
            {"messages": [("user", user_input)]},
            config=config
        )

        # Extract and print the final AI text response
        # response["messages"] contains the full conversation history. 
        # The last element [-1] is always the final answer to the user.
        final_answer = response["messages"][-1].content
        
        print(f"Bot: {final_answer}")

if __name__ == "__main__":
    run_chatbot()

🤖 Smart CLI Chatbot Initialized (LangChain + Llama 3.2)
Tools managed automatically: Math, Weather, Wikipedia, Movies
Type 'exit' or 'quit' to end the conversation.

Bot: {}

(Note: Since the functions don't have any predefined values, the JSON response will be empty.)
Bot: The current temperature in London is 13.3°C. How can I help you further?
Bot: Hello Shaurya! It's nice to meet you. I came across some information about a person named Shaurya in Wikipedia. However, it seems that the article is not very detailed.

If you'd like, we could try searching for more specific information about yourself, such as your interests or accomplishments. Or, if you're feeling adventurous, we could play a game or have a fun conversation!

What sounds interesting to you, Shaurya?
Bot: Sweden has a rich history and culture! Here's an interesting fact: Did you know that Sweden is home to the Abisko National Park, which offers one of the best opportunities in the world to see the Northern Lights? The pa